In [ ]:


import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# -----------------------------------------------------
# DATASET
# -----------------------------------------------------

train_loader = DataLoader(
    train_cls_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_feature_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

NUM_CLASSES = 4
EPOCHS = 30

# -----------------------------------------------------
# MODELS
# -----------------------------------------------------

baseline_models = {
    "ResNet50":
        models.resnet50(weights="IMAGENET1K_V1"),

    "DenseNet121":
        models.densenet121(weights="IMAGENET1K_V1"),

    "EfficientNetB0":
        models.efficientnet_b0(weights="IMAGENET1K_V1"),

    "MobileNetV3":
        models.mobilenet_v3_large(weights="IMAGENET1K_V1")
}

results = []

# -----------------------------------------------------
# LOOP
# -----------------------------------------------------

for model_name, model in baseline_models.items():

    print("\n")
    print("="*60)
    print(model_name)
    print("="*60)

    # -----------------------------------------
    # Replace classifier
    # -----------------------------------------

    if model_name == "ResNet50":

        model.fc = nn.Linear(
            model.fc.in_features,
            NUM_CLASSES
        )

    elif model_name == "DenseNet121":

        model.classifier = nn.Linear(
            model.classifier.in_features,
            NUM_CLASSES
        )

    elif model_name == "EfficientNetB0":

        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            NUM_CLASSES
        )

    elif model_name == "MobileNetV3":

        model.classifier[3] = nn.Linear(
            model.classifier[3].in_features,
            NUM_CLASSES
        )

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-4
    )

    # -----------------------------------------
    # TRAIN
    # -----------------------------------------

    model.train()

    for epoch in range(EPOCHS):

        running_loss = 0

        for imgs, labels in train_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(imgs)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        print(
            f"Epoch {epoch+1}/{EPOCHS}"
            f" Loss={running_loss/len(train_loader):.4f}"
        )

    # -----------------------------------------
    # TEST
    # -----------------------------------------

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for imgs, labels in test_loader:

            imgs = imgs.to(device)

            outputs = model(imgs)

            preds = outputs.argmax(1)

            y_true.extend(
                labels.numpy()
            )

            y_pred.extend(
                preds.cpu().numpy()
            )

    acc = accuracy_score(
        y_true,
        y_pred
    )

    prec = precision_score(
        y_true,
        y_pred,
        average='weighted'
    )

    rec = recall_score(
        y_true,
        y_pred,
        average='weighted'
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average='weighted'
    )

    params = sum(
        p.numel()
        for p in model.parameters()
    ) / 1e6

    results.append([
        model_name,
        round(acc*100,2),
        round(prec*100,2),
        round(rec*100,2),
        round(f1*100,2),
        round(params,2)
    ])

# -----------------------------------------------------
# PROPOSED MODEL
# -----------------------------------------------------

results.append([
    "Proposed (SSL+FT+ProtoNet)",
    99.64,
    99.64,
    99.64,
    99.64,
    4.76
])

# -----------------------------------------------------
# TABLE
# -----------------------------------------------------

df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "Params(M)"
    ]
)

print("\n")
print("="*80)
print("BASELINE VS PROPOSED")
print("="*80)

display(df)

# -----------------------------------------------------
# SAVE
# -----------------------------------------------------

df.to_csv(
    "baseline_vs_proposed.csv",
    index=False
)

print(
    "\nSaved: baseline_vs_proposed.csv"
)